# 06. Factorial

Instead of testing one factor at a time, a **factorial** design varies several factors at once and estimates, in one shot, the **main effects** and the **interactions**. This is where the library meets classical DoE (Fisher, Box).

> Mind the vocabulary: here "interaction" is an **estimated effect** (`A:B`), not the cell-mean *interaction plot* of process optimization.

## The experiment

Two binary factors, `A` and `B`. The outcome depends on each and on their combination (there is an interaction). `FactorialDesign` requires `n_per_cell * 2^K` units; with `K=2` and `n_per_cell=100`, that is 400.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import FactorialAssignment
from skxperiments.design.factorial import FactorialDesign

rng = np.random.default_rng(13)
n_per_cell = 100
n = n_per_cell * 4

design = FactorialDesign(factors=["A", "B"], n_per_cell=n_per_cell, seed=13)
assignment = design.randomize(pd.DataFrame({"unit": np.arange(n)}))

A = assignment.data_["A"].to_numpy()
B = assignment.data_["B"].to_numpy()
y = 2.0 * A + 1.0 * B + 1.0 * (A * B) + rng.normal(size=n) * 0.3

data = assignment.data_.copy()
data["y"] = y
assignment = FactorialAssignment(
    data=data,
    design=design,
    factor_cols=assignment.factor_cols,
    cell_sizes=assignment.cell_sizes_,
    seed=13,
)

## Estimate all effects

`FactorialEstimator` returns the `2^K - 1` contrasts: main effects (`A`, `B`) and the interaction (`A:B`).

In [ ]:
from skxperiments.estimators.factorial_estimator import FactorialEstimator

result = FactorialEstimator(outcome_col="y").fit(assignment).estimate()
for key, value in result.effects.items():
    print(f"{':'.join(key)}: {value:.3f}")

The main effects of `A` and `B` show up clearly, and the `A:B` term is non-zero: because there is an interaction, the joint effect differs from the sum of the isolated effects.

## Visualize

`plot_interaction` shows the effects side by side (needs matplotlib, in the `viz` extra).

In [ ]:
from skxperiments.reporting import plot_interaction

ax = plot_interaction(result)
ax.figure

## What we learned

- A single factorial design estimates main effects and interactions of several factors.
- The `A:B` interaction captures how much one factor's effect depends on the level of the other.
- `FactorialEstimator` returns point estimates only; inference on factorial effects is left to v2.

**Next:** `07. Many tests` shows how to correct p-values when there are several effects or experiments.